In [3]:
import os
import json
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["Tweet_Worldcup"]
collection = db["tweets_raw"]

folder_path = r"D:\Tweet Worldcup\raw"

for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        file_path = os.path.join(folder_path, filename)
        docs = []

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    docs.append(json.loads(line))

        if docs:
            collection.insert_many(docs)

        print(f"{filename} importé avec succès, {len(docs)} documents au total")

print("Tous les fichiers ont été importés avec succès !")

raw0.json importé avec succès, 1999 documents au total
raw1.json importé avec succès, 2000 documents au total
raw10.json importé avec succès, 2000 documents au total
raw100.json importé avec succès, 2000 documents au total
raw1000.json importé avec succès, 2000 documents au total
raw1001.json importé avec succès, 2000 documents au total
raw1002.json importé avec succès, 2000 documents au total
raw1003.json importé avec succès, 2000 documents au total
raw1004.json importé avec succès, 2000 documents au total
raw1005.json importé avec succès, 2000 documents au total
raw1006.json importé avec succès, 2000 documents au total
raw1007.json importé avec succès, 2000 documents au total
raw1008.json importé avec succès, 2000 documents au total
raw1009.json importé avec succès, 2000 documents au total
raw101.json importé avec succès, 2000 documents au total
raw1010.json importé avec succès, 2000 documents au total
raw1011.json importé avec succès, 2000 documents au total
raw1012.json importé ave

In [ ]:
# 2. Vérifier que les données ont bien été importées dans MongoDB


# Afficher toutes les bases de données disponibles dans MongoDB
print("Bases disponibles :", client.list_database_names())

# Afficher toutes les collections présentes dans la base Tweet_Worldcup
print("Collections dans Tweet_Worldcup :", db.list_collection_names())

# Redéfinir la collection pour être sûr d'utiliser les données importées
col = db["tweets_raw"]

Bases disponibles : ['Tweet_Worldcup', 'admin', 'config', 'local']
Collections dans Tweet_Worldcup : ['tweets_raw']


In [ ]:
# 3. Compter le nombre total de tweets


n_tweets = col.estimated_document_count()

print(f"Nombre total de tweets : {n_tweets:,}")

Nombre total de tweets : 4,569,999


In [ ]:
# 4. Compter le nombre d'utilisateurs uniques

pipeline_users = [
    {"$group": {"_id": "$user.id"}},
    {"$count": "n_users"}
]

res = list(col.aggregate(pipeline_users, allowDiskUse=True))

if res:
    print(f"Nombre d'utilisateurs uniques : {res[0]['n_users']:,}")
else:
    print("Aucun utilisateur trouvé.")

Nombre d'utilisateurs uniques : 1,843,439


In [ ]:
# 5. Inspecter la structure d'un tweet


sample = col.find_one()

print("Champs racine :", list(sample.keys()))

print("\nChamps de l'objet user :")
print(list(sample.get("user", {}).keys()))

print("\nChamps de entities :")
print(list(sample.get("entities", {}).keys()))

Champs racine : ['_id', 'created_date', 'current_time', 'in_reply_to_status_id_str', 'in_reply_to_status_id', 'created_at', 'in_reply_to_user_id_str', 'source', 'retweeted_status', 'retweet_count', 'retweeted', 'geo', 'filter_level', 'in_reply_to_screen_name', 'is_quote_status', 'id_str', 'in_reply_to_user_id', 'favorite_count', 'id', 'text', 'place', 'lang', 'quote_count', 'favorited', 'coordinates', 'truncated', 'timestamp_ms', 'reply_count', 'entities', 'contributors', 'user']

Champs de l'objet user :
['utc_offset', 'friends_count', 'profile_image_url_https', 'listed_count', 'profile_background_image_url', 'default_profile_image', 'favourites_count', 'description', 'created_at', 'is_translator', 'profile_background_image_url_https', 'protected', 'screen_name', 'id_str', 'profile_link_color', 'translator_type', 'id', 'geo_enabled', 'profile_background_color', 'lang', 'profile_sidebar_border_color', 'profile_text_color', 'verified', 'profile_image_url', 'time_zone', 'url', 'contribut

In [ ]:
# 6. Afficher un extrait du document complet

# Cette étape permet de mieux comprendre la structure interne d'un tweet.
# On affiche seulement les 3000 premiers caractères pour éviter un affichage trop long.

print(json.dumps(sample, default=str, indent=2)[:3000])

{
  "_id": "6a0af389b890b7b8a4807674",
  "created_date": "2018-06-14 04:14:25",
  "current_time": 1528942465098,
  "in_reply_to_status_id_str": null,
  "in_reply_to_status_id": null,
  "created_at": "Thu Jun 14 02:14:24 +0000 2018",
  "in_reply_to_user_id_str": null,
  "source": "<a href=\"http://twitter.com/download/iphone\" rel=\"nofollow\">Twitter for iPhone</a>",
  "retweeted_status": {
    "extended_entities": {
      "media": [
        {
          "display_url": "pic.twitter.com/sL9ROvLJSx",
          "indices": [
            106,
            129
          ],
          "sizes": {
            "small": {
              "w": 680,
              "h": 314,
              "resize": "fit"
            },
            "large": {
              "w": 1110,
              "h": 512,
              "resize": "fit"
            },
            "thumb": {
              "w": 150,
              "h": 150,
              "resize": "crop"
            },
            "medium": {
              "w": 1110,
        

In [9]:
# 7. Étudier la couverture temporelle des tweets


# Cette étape permet de connaître la période couverte par les tweets.
# Cela sera utile plus tard pour calculer la fréquence de publication.

pipeline_time = [
    {"$group": {
        "_id": None,
        "min_date": {"$min": "$created_at"},
        "max_date": {"$max": "$created_at"}
    }}
]

time_result = list(col.aggregate(pipeline_time))

print(time_result)

[{'_id': None, 'min_date': 'Fri Jun 15 00:00:00 +0000 2018', 'max_date': 'Thu Jun 14 23:59:59 +0000 2018'}]


In [10]:
# 8. Étudier la distribution du nombre de tweets par utilisateur

# Cette étape permet de savoir combien de tweets chaque utilisateur a publiés.
# C'est important car un utilisateur avec un seul tweet donne peu d'informations
# pour analyser son comportement.

import pandas as pd

pipeline_dist = [
    {"$group": {"_id": "$user.id", "n_tweets": {"$sum": 1}}},
    {"$bucket": {
        "groupBy": "$n_tweets",
        "boundaries": [1, 2, 5, 10, 50, 100, 500, 1000, 100000],
        "default": "other",
        "output": {"count": {"$sum": 1}}
    }}
]

dist = list(col.aggregate(pipeline_dist, allowDiskUse=True))

pd.DataFrame(dist)

,_id,count
0,1,1211192
1,2,461985
2,5,106994
3,10,58522
4,50,3429
5,100,1287
6,500,22
7,1000,8
